In [ ]:
# Import & Parameters
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = SparkSession.builder.getOrCreate()

# Path del onfiguration file
config_path = "Files/config_ingestion.csv"


In [ ]:
# Create dataframe for configuration file
df_config = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")
    .csv(config_path)
)

display(df_config)

In [ ]:
# Create a filter to select only data settled with CopyMode == "COPY" and Enabled == 1
rows = (
    df_config
    #.filter((df_config.CopyMode == "COPY") & (df_config.Enabled == 1)) # in case I want to push only COPY
    .filter(df_config.SourceType != "SQL Server") # Excluded SQL Server, data already loaded in Tables
    .filter(df_config.Enabled == 1) # Including also shortcut
    .collect()
)

print(f"Files to process: {len(rows)}")

In [ ]:
# Sanitize  data to guarantee Delta tables constraint
import re

def sanitize_column_names(df):
    """
    Replace empty space and invalid characters
    Forbidden characters for Delta tables: spaces , ; { } ( ) \n \t =
    """
    new_columns = []
    for col_name in df.columns:
        clean_name = re.sub(r'[ ,;{}()\n\t=]', '_', col_name)
        # Remove consecutive underscores
        clean_name = re.sub(r'_+', '_', clean_name)
        # Remove leading/trailing underscores
        clean_name = clean_name.strip('_')
        new_columns.append(clean_name)
    
    for old_name, new_name in zip(df.columns, new_columns):
        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    
    return df

In [ ]:
# Transformation in Delta Tables
results = []

for row in rows:
    source_folder   = row["DestinationFolder"]    # Folders in Files (crm/erp)
    object_name     = row["ObjectName"]           # File name
    dest_table      = row["DestinationTable"]     # target table name
    file_format     = row["FileFormat"].lower()
    load_type       = row["LoadType"]

    source_path = f"Files/{source_folder}/{object_name}"
    target_table = dest_table  # Written in Tables/

    print(f"Processing: {source_path} -> Tables/{target_table}")

    try:
        # Reading data from Files
        if file_format == "csv":
            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "true")
                .csv(source_path)
            )
        elif file_format == "parquet":
            df = spark.read.parquet(source_path)
        elif file_format == "json":
            # 1. Try to read JSON as standard standard (JSON Lines / NDJSON)
            try:
                df = spark.read.json(source_path)
                # Force Spark to verify the schema with a cache and a quick action
                df.cache()
                if len(df.columns) == 1 and df.columns[0] == "_corrupt_record":
                    raise Exception("Formato NDJSON non valido, provo con multiline...")
            except Exception:
                # 2. If it fails , try with option multiline (case structured JSON)
                df = (
                    spark.read
                    .option("multiLine", "true")
                    .json(source_path)
                )
                df.cache()
        else:
            raise ValueError(f"FileFormat not manageable: {file_format}")
        
        # Sanitize column names to guarantee Delta compatibility
        df = sanitize_column_names(df)

        # Additional columns for audit
        df = (
            df
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_source_file", lit(object_name))
        )

        # Write as Delta Table
        write_mode = "overwrite" if load_type == "FULL" else "append"

        (
            df.write
            .format("delta")
            .mode(write_mode)
            .option("overwriteSchema", "true" if write_mode == "overwrite" else "false")
            .saveAsTable(target_table)
        )

        results.append({"table": target_table, "status": "SUCCESS", "rows": df.count()})
        print(f"  ✔ OK — {df.count()} righe scritte in {target_table}")

    except Exception as e:
        results.append({"table": target_table, "status": "FAILED", "error": str(e)})
        print(f"  ✘ ERROR on {target_table}: {e}")

In [ ]:
# Check Failure
df_results = spark.createDataFrame(results)
display(df_results)

# In case of errors it is possible to let fail the pipeline:
failed = [r for r in results if r["status"] == "FAILED"]
if failed:
    raise Exception(f"{len(failed)} tabelle fallite: {[f['table'] for f in failed]}")